# Task 10: Data Preprocessing & Feature Engineering

## Introduction

Data preprocessing and feature engineering are crucial steps in preparing data for ML models. This notebook covers common challenges and techniques.

## 1. Import Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

## 2. Handling Missing Values

In [2]:
# Create sample dataset with missing values
np.random.seed(42)
data = {
    'Age': [25, 30, np.nan, 45, 35, np.nan, 50, 28, 33, 40],
    'Salary': [50000, 60000, 55000, np.nan, 70000, 45000, np.nan, 52000, 58000, 65000],
    'Department': ['IT', 'HR', 'IT', 'Sales', np.nan, 'HR', 'IT', 'Sales', 'HR', 'IT'],
    'Experience': [2, 5, 3, 10, 7, 1, 15, 4, 6, 9]
}
df = pd.DataFrame(data)

print("Original Data with Missing Values:")
print(df)
print(f"\nMissing values per column:\n{df.isnull().sum()}")

Original Data with Missing Values:
    Age   Salary Department  Experience
0  25.0  50000.0         IT           2
1  30.0  60000.0         HR           5
2   NaN  55000.0         IT           3
3  45.0      NaN      Sales          10
4  35.0  70000.0        NaN           7
5   NaN  45000.0         HR           1
6  50.0      NaN         IT          15
7  28.0  52000.0      Sales           4
8  33.0  58000.0         HR           6
9  40.0  65000.0         IT           9

Missing values per column:
Age           2
Salary        2
Department    1
Experience    0
dtype: int64


### Why Mean/Median Imputation?

Mean imputation is suitable when data is **MCAR (Missing Completely at Random)** and has no extreme outliers. Median is preferred when data has **outliers** as it's more robust to skewness.

In [3]:
# Handle missing values
# For numerical: use median (robust to outliers)
df_cleaned = df.copy()
df_cleaned['Age'] = df_cleaned['Age'].fillna(df_cleaned['Age'].median())
df_cleaned['Salary'] = df_cleaned['Salary'].fillna(df_cleaned['Salary'].median())

# For categorical: use mode
df_cleaned['Department'] = df_cleaned['Department'].fillna(df_cleaned['Department'].mode()[0])

print("Data after Missing Value Imputation:")
print(df_cleaned)

Data after Missing Value Imputation:
    Age   Salary Department  Experience
0  25.0  50000.0         IT           2
1  30.0  60000.0         HR           5
2  34.0  55000.0         IT           3
3  45.0  56500.0      Sales          10
4  35.0  70000.0         IT           7
5  34.0  45000.0         HR           1
6  50.0  56500.0         IT          15
7  28.0  52000.0      Sales           4
8  33.0  58000.0         HR           6
9  40.0  65000.0         IT           9


## 3. Feature Scaling

In [4]:
# Create sample data with different scales
np.random.seed(42)
X = np.random.rand(100, 3) * np.array([1000, 10, 100])
df_scaled = pd.DataFrame(X, columns=['Income', 'Age', 'Score'])

print("Original Data (different scales):")
print(df_scaled.describe())

Original Data (different scales):
           Income         Age       Score
count  100.000000  100.000000  100.000000
mean   476.849233    5.124614   49.630319
std    285.957102    3.102190   28.811399
min      5.522117    0.050616    0.695213
25%    238.588760    2.533467   23.754043
50%    472.761372    5.205169   52.241223
75%    713.108520    7.812006   71.254455
max    990.053850    9.856505   96.990985


### Why StandardScaler?

StandardScaler (z-score normalization) is used when algorithm assumes **zero-centered data** or uses **distance-based metrics** (KNN, SVM, Neural Networks). It transforms data to mean=0, std=1.

In [5]:
# StandardScaler - transforms to mean=0, std=1
scaler_standard = StandardScaler()
X_standard = scaler_standard.fit_transform(df_scaled)

print("After StandardScaler:")
print(pd.DataFrame(X_standard, columns=df_scaled.columns).describe().round(2))

After StandardScaler:
       Income     Age   Score
count  100.00  100.00  100.00
mean    -0.00    0.00    0.00
std      1.01    1.01    1.01
min     -1.66   -1.64   -1.71
25%     -0.84   -0.84   -0.90
50%     -0.01    0.03    0.09
75%      0.83    0.87    0.75
max      1.80    1.53    1.65


### Why MinMaxScaler?

MinMaxScaler is used when we need **bounded values** (0-1 range) or for **algorithms that don't assume normality** like Neural Networks and K-Nearest Neighbors.

In [6]:
# MinMaxScaler - transforms to range [0, 1]
scaler_minmax = MinMaxScaler()
X_minmax = scaler_minmax.fit_transform(df_scaled)

print("After MinMaxScaler:")
print(pd.DataFrame(X_minmax, columns=df_scaled.columns).describe().round(2))

After MinMaxScaler:
       Income     Age   Score
count  100.00  100.00  100.00
mean     0.48    0.52    0.51
std      0.29    0.32    0.30
min      0.00    0.00    0.00
25%      0.24    0.25    0.24
50%      0.47    0.53    0.54
75%      0.72    0.79    0.73
max      1.00    1.00    1.00


## 4. Encoding Categorical Variables

In [7]:
# Sample categorical data
df_cat = pd.DataFrame({
    'Color': ['Red', 'Blue', 'Green', 'Red', 'Blue'],
    'Size': ['S', 'M', 'L', 'S', 'M'],
    'Label': [1, 0, 1, 0, 1]
})

print("Categorical Data:")
print(df_cat)

Categorical Data:
   Color Size  Label
0    Red    S      1
1   Blue    M      0
2  Green    L      1
3    Red    S      0
4   Blue    M      1


### Why Label Encoding?
Label Encoding converts categories to numbers (0, 1, 2...). It's suitable for **ordinal variables** or when the algorithm can learn **ordinal relationships** (Decision Trees, Random Forest).

In [8]:
# Label Encoding - for ordinal/nominal with tree-based algorithms
le = LabelEncoder()
df_cat['Color_encoded'] = le.fit_transform(df_cat['Color'])

print("After Label Encoding:")
print(df_cat)

After Label Encoding:
   Color Size  Label  Color_encoded
0    Red    S      1              2
1   Blue    M      0              0
2  Green    L      1              1
3    Red    S      0              2
4   Blue    M      1              0


### Why One-Hot Encoding?
One-Hot Encoding creates binary columns for each category. It's suitable for **nominal variables** without ordinal relationship and for **linear models** where numbers shouldn't imply order.

In [9]:
# One-Hot Encoding - for nominal variables
df_onehot = pd.get_dummies(df_cat[['Color', 'Size']], drop_first=False)

print("After One-Hot Encoding:")
print(df_onehot)

After One-Hot Encoding:
   Color_Blue  Color_Green  Color_Red  Size_L  Size_M  Size_S
0       False        False       True   False   False    True
1        True        False      False   False    True   False
2       False         True      False    True   False   False
3       False        False       True   False   False    True
4        True        False      False   False    True   False


## 5. Using Pipeline & ColumnTransformer

In [10]:
# Create sample dataset
np.random.seed(42)
X = pd.DataFrame({
    'Age': [25, 30, 35, 40, 45, 50, 22, 28],
    'Income': [50000, 60000, 55000, 70000, 80000, 90000, 45000, 52000],
    'City': ['NYC', 'LA', 'NYC', 'Chicago', 'LA', 'Chicago', 'NYC', 'LA']
})
y = [1, 0, 1, 0, 0, 1, 1, 0]

print("Sample Dataset:")
print(X)

Sample Dataset:
   Age  Income     City
0   25   50000      NYC
1   30   60000       LA
2   35   55000      NYC
3   40   70000  Chicago
4   45   80000       LA
5   50   90000  Chicago
6   22   45000      NYC
7   28   52000       LA


### Why Pipeline?
Pipeline chains multiple transformations and ensures **consistent preprocessing** for both training and test data, prevents **data leakage** by fitting only on training data.

In [11]:
# Define preprocessing for numeric and categorical columns
numeric_features = ['Age', 'Income']
categorical_features = ['City']

numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(drop='first'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# Apply transformation
X_transformed = preprocessor.fit_transform(X)

print("Transformed Data Shape:", X_transformed.shape)
print("\nTransformed Data (first 5 rows):")
print(X_transformed[:5].round(2))

Transformed Data Shape: (8, 4)

Transformed Data (first 5 rows):
[[-1.01 -0.86  0.    1.  ]
 [-0.47 -0.19  1.    0.  ]
 [ 0.07 -0.52  0.    1.  ]
 [ 0.61  0.49  0.    0.  ]
 [ 1.14  1.17  1.    0.  ]]


## 6. Summary

| Technique | When to Use | Why |
|-----------|-------------|-----|
| Mean/Median Imputation | MCAR data, numerical features | Mean for normal data, Median for outliers |
| StandardScaler | Distance-based algorithms | Zero-centered, unit variance |
| MinMaxScaler | Bounded data needed, Neural Networks | Maps to [0,1] range |
| Label Encoding | Ordinal variables, Tree algorithms | Preserves order |
| One-Hot Encoding | Nominal variables, Linear models | No false ordinal relationship |
| Pipeline | Any ML workflow | Prevents leakage, ensures consistency |